# 05 - Comparación de modelos de detección de humo y fuego

Consolida los `metrics_summary.csv` de todos los experimentos y genera la tabla
y las figuras comparativas.

Este notebook **no necesita GPU, ni Drive, ni el dataset**: trabaja solo con los
CSV versionados en el repositorio, así que la comparación es reproducible por
cualquiera que clone el proyecto.

In [ ]:
# ============================================================
# Setup
# ============================================================

from pathlib import Path
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q -r https://raw.githubusercontent.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego/main/requirements.txt
    !git clone -q https://github.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego.git /content/VpC2---Deteccion-de-humo-y-fuego
    PROJECT_DIR = Path("/content/VpC2---Deteccion-de-humo-y-fuego")
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("PROJECT_DIR:", PROJECT_DIR)

In [ ]:
# ============================================================
# Carga de los resúmenes de cada experimento
# ============================================================

import pandas as pd

from src.reporting.summary import load_metrics_summaries

RESULTS_DIR = PROJECT_DIR / "reports" / "results"
df = load_metrics_summaries(RESULTS_DIR)

if df.empty:
    raise RuntimeError(
        f"No se encontró ningún metrics_summary.csv en {RESULTS_DIR}. "
        "Correr antes los notebooks 02, 03 y 04."
    )

print(f"Experimentos encontrados: {len(df)}")
display(df)

In [ ]:
# ============================================================
# Tabla comparativa principal
# ============================================================

COLUMNAS = [
    "experiment", "family", "params_M", "epochs", "train_time_min",
    "mAP50", "mAP50_95", "precision", "recall", "f1", "fps",
]

tabla = df[COLUMNAS].copy()
tabla.columns = [
    "Experimento", "Familia", "Params (M)", "Épocas", "Entrenamiento (min)",
    "mAP@0.5", "mAP@0.5:0.95", "Precisión", "Recall", "F1", "FPS",
]

display(tabla.style.background_gradient(subset=["mAP@0.5", "mAP@0.5:0.95"], cmap="Greens"))

In [ ]:
# ============================================================
# Desempeño por clase
# ============================================================

por_clase = df[
    ["experiment", "mAP50_smoke", "mAP50_fire", "mAP50_95_smoke", "mAP50_95_fire"]
].copy()
por_clase.columns = [
    "Experimento", "mAP@0.5 smoke", "mAP@0.5 fire",
    "mAP@0.5:0.95 smoke", "mAP@0.5:0.95 fire",
]

display(por_clase)

# El humo suele ser más difícil que el fuego: bordes difusos y sin forma definida.
diferencia = (df["mAP50_fire"] - df["mAP50_smoke"]).mean()
print(f"\nVentaja media de fire sobre smoke en mAP@0.5: {diferencia:+.4f}")

In [ ]:
# ============================================================
# Figuras comparativas
# ============================================================

from IPython.display import Image, display

from src.reporting.plots import plot_model_comparison

FIGURES_DIR = PROJECT_DIR / "reports" / "figures" / "comparacion"
rutas = plot_model_comparison(df, FIGURES_DIR)

for ruta in rutas:
    print(ruta.name)
    display(Image(filename=str(ruta)))

In [ ]:
# ============================================================
# Guardar la tabla consolidada
# ============================================================

COMPARISON_CSV = PROJECT_DIR / "reports" / "results" / "comparacion_modelos.csv"
df.to_csv(COMPARISON_CSV, index=False)

print("Tabla comparativa guardada en:", COMPARISON_CSV)

mejor = df.iloc[0]
print(
    f"\nMejor modelo por mAP@0.5: {mejor['experiment']} "
    f"({mejor['mAP50']:.4f}), a {mejor['fps']:.1f} FPS."
)

In [ ]:
# ============================================================
# Commit de la comparación
# ============================================================

import subprocess

%cd {PROJECT_DIR}

!git config user.name "Gabriela-Sol"
!git config user.email "solgab.salazar@gmail.com"

for path in ["reports/figures/comparacion/", "reports/results/comparacion_modelos.csv"]:
    if Path(path).exists():
        subprocess.run(["git", "add", path], check=True)
        print("Agregado:", path)

status = subprocess.run(["git", "status", "--short"], text=True, capture_output=True)
print(status.stdout)

if status.stdout.strip():
    subprocess.run(["git", "commit", "-m", "results: comparacion entre modelos"], check=True)
    print("Commit creado. Para publicarlo: !git push origin main")
else:
    print("No hay cambios nuevos para commitear.")